In [44]:
from typing import TypedDict

import httpx
from minsearch import Index
from gitsource import GithubRepositoryDataReader


# Define a TypedDict for the parsed document
class Book(TypedDict):
    title: str
    book_url: str
    pdf_url: str
    

# Read the documentation files from the GitHub repository
reader = GithubRepositoryDataReader(
    repo_owner="alexeygrigorev",
    repo_name="ai-engineering-buildcamp-code",
    allowed_extensions={"csv"},
)
files = reader.read()

In [5]:
# Parse the files
parsed_books = [Book(**doc.parse()) for doc in files if doc.filename == "01-foundation/homework/books.csv"]

In [7]:
import csv
from pathlib import Path


def download_book(url: str) -> bytes:
    """
    Downloads the book file from a GitHub repository.

    Args:
        url (str): The URL of the book file.

    Returns:
        bytes: The content of the book file.
    """

    with httpx.Client(follow_redirects=True) as client:
        response = client.get(url)
        response.raise_for_status()
        return response.content


# Save PDFs to the books/ directory
books_dir = Path("books")
books_dir.mkdir(exist_ok=True)

# Download every PDF URL from the parsed books CSV
for book in parsed_books:
    rows = csv.DictReader(book.get("content", "").splitlines())
    for row in rows:
        pdf_url = row.get("pdf_url", "")
        if pdf_url.endswith(".pdf"):
            filename = pdf_url.rsplit("/", 1)[-1]
            content = download_book(pdf_url)
            (books_dir / filename).write_bytes(content)
            print(f"Downloaded {filename} to {books_dir}/")

Downloaded thinkpython2.pdf to books/
Downloaded thinkdsp.pdf to books/
Downloaded thinkcomplexity2.pdf to books/
Downloaded thinkjava2.pdf to books/
Downloaded PhysicalModelingInMatlab4.pdf to books/
Downloaded thinkos.pdf to books/
Downloaded Think-C.pdf to books/


In [11]:
from markitdown import MarkItDown
from anthropic import Anthropic


# Initialize MarkItDown with the Anthropic LLM client
md = MarkItDown(
    enable_plugins=True,
    llm_client=Anthropic(),
    llm_model="claude-opus-5",
)

# Save the converted markdown files to the books_text/ directory
books_text_dir = Path("books_text")
books_text_dir.mkdir(exist_ok=True)

for file in books_dir.iterdir():
    print(f"Converting file {file.name} to markdown...")
    result = md.convert(file)
    (books_text_dir / f"{file.stem}.md").write_text(result.markdown)

Converting file thinkos.pdf to markdown...
Converting file thinkpython2.pdf to markdown...
Converting file Think-C.pdf to markdown...
Converting file thinkcomplexity2.pdf to markdown...
Converting file PhysicalModelingInMatlab4.pdf to markdown...
Converting file thinkdsp.pdf to markdown...
Converting file thinkjava2.pdf to markdown...


In [12]:
# Count the total number of lines in all markdown files
total_lines = 0
for file in books_text_dir.iterdir():
    lines = file.read_text().splitlines()
    total_lines += len(lines)
    print(f"{file.name}: {len(lines)} lines")

print(f"Total lines of text in all markdown files: {total_lines}")

thinkpython2.md: 9703 lines
thinkos.md: 3541 lines
Think-C.md: 5502 lines
thinkjava2.md: 12890 lines
thinkcomplexity2.md: 7441 lines
thinkdsp.md: 5705 lines
PhysicalModelingInMatlab4.md: 6155 lines
Total lines of text in all markdown files: 50937


In [43]:
# Read each markdown file, remove empty/whitespace-only lines,
# and store the cleaned lines in a dictionary.
processed = {}
processed_books = []

for file in books_text_dir.iterdir():
    lines = [line for line in file.read_text().splitlines() if line.strip()]
    processed["source"] = file.name
    processed["content"] = lines
    processed_books.append(processed.copy())

total_non_empty_lines = sum(len(lines) for lines in [book["content"] for book in processed_books])
print(f"Total non-empty lines in all markdown files: {total_non_empty_lines}")

Total non-empty lines in all markdown files: 49544


In [34]:
def sliding_window(text: str, size: int = 1000, step: int = 500):
    start = 0
    chunks = []
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break

    return chunks

In [40]:
for book in processed_books:
    source = book["source"]
    content = book["content"]
    chunks = sliding_window(content, 100, 50)
    print(f"Book: {source}, Number of chunks: {len(chunks)}")

Book: thinkpython2.md, Number of chunks: 189
Book: thinkos.md, Number of chunks: 68
Book: Think-C.md, Number of chunks: 107
Book: thinkjava2.md, Number of chunks: 250
Book: thinkcomplexity2.md, Number of chunks: 144
Book: thinkdsp.md, Number of chunks: 111
Book: PhysicalModelingInMatlab4.md, Number of chunks: 119


In [62]:
def prepare_documents(chunks: list[dict]) -> list[dict]:
    """
    Prepares documents for indexing by converting lists of strings into a single string.

    Args:
        chunks (list[dict]): A list of dictionaries representing the content of the document.
    
    Returns:
        list[dict]: A list of dictionaries representing the prepared documents.
    """

    documents = []
    for book in processed_books:
        source = book["source"]
        chunks = sliding_window(book["content"], 100, 50)
        for chunk in chunks:
            documents.append({
                "source": source,
                "content": "\n".join(chunk["content"]),
            })
    return documents


# Prepare the documents for indexing
documents = prepare_documents(chunks)
print(f"Prepared {len(documents)} documents for indexing.")

# Index the documents using MinSearch
index = Index(text_fields=["content"])
index.fit(documents)

Prepared 988 documents for indexing.


In [63]:
results = index.search("python function definition", num_results=5)

In [64]:
results

[{'source': 'PhysicalModelingInMatlab4.md',
  'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfere with\nvariables

In [83]:
import json
from typing import Literal

from pydantic import BaseModel, Field


client = Anthropic()


instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")


def build_prompt(question: str, search_results: list[dict]) -> str:
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt


def search(question: str) -> list[dict]:
    return index.search(question, num_results=5)


def llm(
    user_prompt: str,
    instructions: str,
    max_tokens: int = 2056,
    model='claude-opus-5-5'
):
    response = client.messages.create(
        model=model,
        messages=[
            {"role": "assistant", "content": instructions},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        thinking={"type": "adaptive", "display": "summarized"},
    )
    return response


def llm_structured(
    user_prompt: str,
    instructions: str,
    max_tokens: int = 2056,
    model='claude-opus-5-5'
):
    response = client.messages.parse(
        model=model,
        messages=[
            {"role": "assistant", "content": instructions},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_tokens,
        output_format=RAGResponse,
        thinking={"type": "adaptive", "display": "summarized"},
    )
    return response


def rag(query: str, use_structured_output: bool = True) -> dict:
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    if use_structured_output:
        response = llm_structured(prompt, instructions)
    else:
        response = llm(prompt, instructions)

    answer = ""
    for block in response.content:
        if block.type == "text":
            answer += block.text

    return {"answer": answer, "usage": {
        "inputs": response.usage.input_tokens,
        "outputs": response.usage.output_tokens,
    }}

In [84]:
outputs = rag("python function definition")
structured_outputs = rag("python function definition", use_structured_output=True)
print("RAG Answer:", outputs["answer"])
print("RAG Answer (structured):", structured_outputs["answer"])
print("RAG Usage:", outputs["usage"])
print("RAG Usage (structured):", structured_outputs["usage"])

RAG Answer: {"answer":"The course materials provided don't actually show how to define a function in Python. The only Python-related mention is in *Think Python* (Chapter 1), which notes that the parentheses in `print('Hello, World!')` indicate that `print` is a function, and that functions are covered in Chapter 3. That chapter isn't in the excerpts I have.\n\nWhat the materials do cover in detail is **function definition in MATLAB** (*Physical Modeling in MATLAB*, Chapter 5). Several of its concepts carry over to Python:\n\n- **Signature:** The first line of a function definition is its signature. It names the function and specifies its inputs and outputs. In MATLAB this looks like `function res = myfunc(x)`.\n- **Own workspace (local variables):** Each function has its own workspace. Variables created inside it exist only while the function runs and don't collide with variables of the same name elsewhere. This is the main reason to use functions instead of scripts, because it avoids